# Rule Support & Confidence Counting — Original Train Benchmark

Dataset: `body_test_data/original_train.ttl`

| Rules file | Rules | Used for |
|---|---|---|
| `original_train_rules_50safe.txt` | 50 (9×1atom + 19×2atom + 22×3atom) | Rewritten C++, Optimized CPU, GPU — streaming |
| `original_train_rules_300minutes.txt` | 931 (9×1atom + 96×2atom + 826×3atom) | Full/original long-running reference |

The 50-rule safe sample is the existing 30-rule sample plus 10 deliberately low-combination additions. The additions avoid `rdf:type` and use only low-fanout ontology-style body predicates. Each implementation streams output per rule as it finishes.

In [ ]:
import os, subprocess, sys, time, json, re, statistics
from pathlib import Path
from collections import defaultdict

# ═════════════════════════════════════════════════════════════════════════════
# CHANGE THIS to wherever you uploaded / cloned the MasterThesisFinal folder.
#   Colab default : /content/drive/MyDrive/MasterThesisFinal
#   VSE Jupyter   : /home/jupyter-naav00@vse.cz/MasterThesisFinal
BASE_PATH = '/home/jupyter-naav00@vse.cz/MasterThesisFinal'
# ═════════════════════════════════════════════════════════════════════════════

BASE = Path(BASE_PATH)
if not BASE.exists():
    raise FileNotFoundError(f'Folder not found: {BASE}\nUpdate BASE_PATH above.')
if not (BASE / 'GPU_03').exists():
    raise FileNotFoundError(f'Cannot find GPU_03 inside {BASE}')

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

def has_command(cmd):
    return subprocess.run(['which', cmd], capture_output=True).returncode == 0

HAS_NVCC = has_command('nvcc')
HAS_GPU  = False
if HAS_NVCC:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                       capture_output=True, text=True)
    HAS_GPU = r.returncode == 0 and r.stdout.strip() != ''

print(f'Environment : {"Google Colab" if IS_COLAB else "JupyterHub / local"}')
print(f'nvcc found  : {HAS_NVCC}')
print(f'GPU found   : {HAS_GPU}')
if HAS_GPU:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap',
                          '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())

DIR_REWRITTEN = BASE / 'Rewritten_CPP'
DIR_CPU       = BASE / 'CPU_01_nonparallel'
DIR_GPU       = BASE / 'GPU_03'
DATA_DIR      = BASE / 'body_test_data'

TTL_FILE      = DATA_DIR / 'original_train.ttl'
RULES_50SAFE      = DATA_DIR / 'original_train_rules_50sample.txt'
RULES_FULL    = DATA_DIR / 'original_train_rules_300minutes.txt'
RULES_50SAFE_JSON = DATA_DIR / 'original_train_rules_50sample.json'

print(f'\nBase dir   : {BASE}')
print(f'TTL file   : {TTL_FILE}  (exists={TTL_FILE.exists()})')
print(f'Rules 50 safe: {RULES_50SAFE}  (exists={RULES_50SAFE.exists()})')
print(f'Rules 50 safe json: {RULES_50SAFE_JSON}  (exists={RULES_50SAFE_JSON.exists()})')
print(f'Rules full : {RULES_FULL}  (exists={RULES_FULL.exists()})')

In [ ]:
def run(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result

def benchmark(cmd, cwd=None, runs=3):
    times = []
    last_stdout = ''
    for i in range(runs):
        t0 = time.perf_counter()
        r  = subprocess.run(cmd, shell=True, cwd=cwd or BASE,
                            capture_output=True, text=True)
        t1 = time.perf_counter()
        if r.returncode != 0:
            raise RuntimeError(f'Benchmark failed: {r.stderr}')
        times.append((t1 - t0) * 1000)
        last_stdout = r.stdout
    return min(times), sum(times)/len(times), last_stdout

def extract_total_ms(stdout):
    for pattern in [r'Total wall time:\s+([0-9.]+)\s*ms',
                    r'Total time:\s+([0-9.]+)\s*ms']:
        m = re.search(pattern, stdout)
        if m: return float(m.group(1))
    return None

def count_body_atoms(rule_text):
    return rule_text.split('=>')[0].count('^') + 1

def parse_rewritten_output(stdout):
    results = []
    lines = stdout.split('\n')
    i = 0
    while i < len(lines):
        m = re.match(r'Rule \d+: (.+)', lines[i])
        if m:
            rule_text = m.group(1)
            j = i + 1
            while j < len(lines) and 'support=' not in lines[j]:
                j += 1
            if j < len(lines):
                sm = re.search(r'supportMs=([0-9.]+)', lines[j])
                cm = re.search(r'confidenceMs=([0-9.]+)', lines[j])
                if sm and cm:
                    results.append({
                        'body_atoms':    count_body_atoms(rule_text),
                        'supportMs':     float(sm.group(1)),
                        'confidenceMs':  float(cm.group(1))
                    })
            i = j + 1
        else:
            i += 1
    return results

def parse_cpu_json(stdout):
    data = json.loads(stdout)
    return [{
        'body_atoms':    count_body_atoms(r['rule']),
        'supportMs':     r['supportMs'],
        'confidenceMs':  r['confidenceMs']
    } for r in data]

def print_body_stats(records, label):
    groups = defaultdict(list)
    for r in records:
        groups[r['body_atoms']].append(r)
    total_sup  = sum(r['supportMs']    for r in records)
    total_conf = sum(r['confidenceMs'] for r in records)
    print(f'\n=== {label} — Statistics by body size ===')
    print(f'Total support time:    {total_sup/1000:.2f} s')
    print(f'Total confidence time: {total_conf/1000:.2f} s')
    for n in sorted(groups.keys()):
        g = groups[n]
        sup  = [r['supportMs']    for r in g]
        conf = [r['confidenceMs'] for r in g]
        print(f'\nBody size {n} ({len(g)} rules):')
        print(f'  Support:    avg={statistics.mean(sup):.1f} ms  median={statistics.median(sup):.1f} ms  '
              f'min={min(sup):.1f} ms  max={max(sup):.1f} ms')
        print(f'  Confidence: avg={statistics.mean(conf):.1f} ms  median={statistics.median(conf):.1f} ms  '
              f'min={min(conf):.1f} ms  max={max(conf):.1f} ms')


def benchmark_stream(cmd, cwd=None):
    """Run cmd once, printing each line in real-time. Returns (wall_ms, stdout)."""
    collected = []
    t0 = time.perf_counter()
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or BASE,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
        collected.append(line)
    proc.wait()
    t1 = time.perf_counter()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed (exit {proc.returncode})')
    return (t1 - t0) * 1000, ''.join(collected)


import psutil, threading

def get_vram_mb():
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
        capture_output=True, text=True)
    if r.returncode != 0: return 0
    return int(r.stdout.strip().split('\n')[0])

def measure_peak_memory(cmd, cwd=None, interval=0.5):
    baseline_ram  = psutil.virtual_memory().used / (1024*1024)
    baseline_vram = get_vram_mb()
    peak_ram  = [baseline_ram]
    peak_vram = [baseline_vram]
    stop_flag = [False]
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or BASE,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    def sampler():
        while not stop_flag[0]:
            peak_ram[0]  = max(peak_ram[0],  psutil.virtual_memory().used / (1024*1024))
            peak_vram[0] = max(peak_vram[0], get_vram_mb())
            time.sleep(interval)
    t = threading.Thread(target=sampler, daemon=True)
    t0 = time.perf_counter()
    t.start()
    for line in proc.stdout:
        pass  # drain output silently
    proc.wait()
    stop_flag[0] = True
    t.join()
    wall_ms = (time.perf_counter() - t0) * 1000
    return peak_ram[0] - baseline_ram, peak_vram[0] - baseline_vram, wall_ms

print('Helpers ready.')
def parse_gpu_output(stdout):
    """Parse per-rule timing from GPU bench_gpu output."""
    results = []
    for line in stdout.splitlines():
        # line format: "Rule N: <rule text> | ... SupportTime: X ms, ConfidenceTime: Y ms"
        if not line.startswith('Rule '):
            continue
        pipe = line.find('|')
        if pipe == -1:
            continue
        rule_text = line[line.index(':')+1:pipe].strip()
        sm = re.search(r'SupportTime:\s*([0-9.]+)', line)
        cm = re.search(r'ConfidenceTime:\s*([0-9.]+)', line)
        if sm and cm:
            results.append({
                'body_atoms':   count_body_atoms(rule_text),
                'supportMs':    float(sm.group(1)),
                'confidenceMs': float(cm.group(1)),
            })
    return results


---
## Step 1 — Compile all implementations

In [ ]:
print('Compiling Rewritten_CPP ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_rewritten',
    cwd=DIR_REWRITTEN
)
print('OK — binary: Rewritten_CPP/rdf_rules_rewritten')

In [ ]:
print('Compiling CPU_01_nonparallel ...')
run(
    'g++ -O3 -std=c++17 '
    'main.cpp FinalRule.cpp RdfIndexes.cpp RuleParser.cpp SupportCounting.cpp '
    '-o rdf_rules_test',
    cwd=DIR_CPU
)
print('OK — binary: CPU_01_nonparallel/rdf_rules_test')

In [ ]:
if not HAS_NVCC:
    print('nvcc not found — skipping GPU compilation.')
else:
    print('Compiling GPU_03 benchmark binary ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets '
        'bench_gpu.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o bench_gpu',
        cwd=DIR_GPU
    )
    print('OK — binary: GPU_03/bench_gpu')

    print('\nCompiling GPU_03 shared library ...')
    run(
        'nvcc -O3 -std=c++17 -Wno-deprecated-gpu-targets -shared -Xcompiler -fPIC '
        'gpu_rules_lib.cu FinalRule.cpp RdfIndexes.cpp RuleParser.cpp '
        '-o libgpu_rules.so',
        cwd=DIR_GPU
    )
    print('OK — library: GPU_03/libgpu_rules.so')

---
## Step 2 — Rewritten C++ (50-rule safe sample, streaming)

In [ ]:
print('Running Rewritten_CPP on 50-rule safe sample (streaming output) ...')
cmd_rewritten = f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULES_50SAFE_JSON}'
wall_ms_r, out_r = benchmark_stream(cmd_rewritten)
total_r = extract_total_ms(out_r)
print(f'\nWall time: {wall_ms_r:.0f} ms  (reported total={total_r} ms)')

records_r = parse_rewritten_output(out_r)
print_body_stats(records_r, 'Rewritten C++ (50 safe rules)')


---
## Step 3 — Optimized CPU (50-rule safe sample, streaming)

In [ ]:
print('Running CPU_01_nonparallel on 50-rule safe sample (streaming output) ...')
cmd_cpu = f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULES_50SAFE_JSON}'
wall_ms_c, out_c = benchmark_stream(cmd_cpu)
total_c = extract_total_ms(out_c)
print(f'\nWall time: {wall_ms_c:.0f} ms  (reported total={total_c} ms)')

print('\nCollecting per-rule data (--json) ...')
r_json = subprocess.run(
    f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULES_50SAFE_JSON} --json',
    shell=True, capture_output=True, text=True
)
records_c = parse_cpu_json(r_json.stdout)
print_body_stats(records_c, 'Optimized CPU (50 safe rules)')


---
## Step 4 — GPU (50-rule safe sample, streaming)

In [ ]:
min_ms_g = total_g = None
records_g = []

if not HAS_GPU:
    print('No GPU available — skipping.')
else:
    print('Running GPU_03 on 50-rule safe sample (streaming output) ...')
    cmd_gpu = f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULES_50SAFE_JSON}'
    wall_ms_g, out_g = benchmark_stream(cmd_gpu)
    total_g = extract_total_ms(out_g)
    min_ms_g = wall_ms_g
    print(f'\nWall time: {wall_ms_g:.0f} ms  (reported total={total_g} ms)')

    records_g = parse_gpu_output(out_g)
    print_body_stats(records_g, 'GPU CUDA (50 safe rules)')


---
## Step 6 — Results comparison

In [ ]:
from collections import defaultdict

print('Dataset: Original Train  |  50 safe rules (9×1atom + 19×2atom + 22×3atom)\n')

# ── Overall totals ──────────────────────────────────────────────────────────
header = f'{"Implementation":<22} {"Total (ms)":>15} {"Speedup vs Rewritten":>22}'
print(header)
print('-' * len(header))
rows = [
    ('Rewritten C++', f'{wall_ms_r:.0f}',  '1.0x'),
    ('Optimized CPU', f'{wall_ms_c:.0f}',  f'{wall_ms_r/wall_ms_c:.2f}x' if wall_ms_c else 'N/A'),
    ('GPU (CUDA)',    f'{min_ms_g:.0f}' if min_ms_g else 'N/A (no GPU)',
                     f'{wall_ms_r/min_ms_g:.2f}x' if min_ms_g else 'N/A'),
]
for impl, total, speedup in rows:
    print(f'{impl:<22} {total:>15} {speedup:>22}')

# ── Per-body-atom breakdown ─────────────────────────────────────────────────
print('\n── Avg confidence time per body-atom count (ms/rule) ──────────────────')
hdr2 = f'{"Body atoms":<12} {"Rewritten":>12} {"CPU":>12} {"GPU":>12}'
print(hdr2)
print('-' * len(hdr2))

def avg_conf_by_n(records):
    groups = defaultdict(list)
    for r in records:
        groups[r['body_atoms']].append(r['confidenceMs'])
    return {n: sum(v)/len(v) for n, v in groups.items()}

def avg_sup_by_n(records):
    groups = defaultdict(list)
    for r in records:
        groups[r['body_atoms']].append(r['supportMs'])
    return {n: sum(v)/len(v) for n, v in groups.items()}

conf_r = avg_conf_by_n(records_r)
conf_c = avg_conf_by_n(records_c)
conf_g = avg_conf_by_n(records_g) if records_g else {}

all_ns = sorted(set(conf_r) | set(conf_c) | set(conf_g))
for n in all_ns:
    r_v = f'{conf_r.get(n, 0):.1f}'
    c_v = f'{conf_c.get(n, 0):.1f}'
    g_v = f'{conf_g.get(n, 0):.1f}' if conf_g else 'N/A'
    print(f'{n:<12} {r_v:>12} {c_v:>12} {g_v:>12}')

print('\n── Avg support time per body-atom count (ms/rule) ─────────────────────')
print(hdr2)
print('-' * len(hdr2))
sup_r = avg_sup_by_n(records_r)
sup_c = avg_sup_by_n(records_c)
sup_g = avg_sup_by_n(records_g) if records_g else {}
for n in all_ns:
    r_v = f'{sup_r.get(n, 0):.2f}'
    c_v = f'{sup_c.get(n, 0):.2f}'
    g_v = f'{sup_g.get(n, 0):.2f}' if sup_g else 'N/A'
    print(f'{n:<12} {r_v:>12} {c_v:>12} {g_v:>12}')


---
## Memory Usage — Peak RAM and VRAM per implementation

In [ ]:
print('Measuring peak memory usage (50-rule safe sample, runs silently) ...\n')

ram_r, vram_r, _ = measure_peak_memory(f'{DIR_REWRITTEN}/rdf_rules_rewritten {TTL_FILE} {RULES_50SAFE_JSON}')
print(f'Rewritten C++  — RAM: {ram_r:>8,.0f} MB   VRAM: {vram_r:>6,.0f} MB')

ram_c, vram_c, _ = measure_peak_memory(f'{DIR_CPU}/rdf_rules_test {TTL_FILE} {RULES_50SAFE_JSON}')
print(f'Optimized CPU  — RAM: {ram_c:>8,.0f} MB   VRAM: {vram_c:>6,.0f} MB')

if HAS_GPU:
    ram_g, vram_g, _ = measure_peak_memory(f'{DIR_GPU}/bench_gpu {TTL_FILE} {RULES_50SAFE_JSON}')
    print(f'GPU (CUDA)     — RAM: {ram_g:>8,.0f} MB   VRAM: {vram_g:>6,.0f} MB')
else:
    print('GPU (CUDA)     — skipped (no GPU)')